# Entrenamiento del Modelo de Clasificación de Residuos
## Tacho IA - MobileNetV2 Transfer Learning

Este notebook entrena un modelo para clasificar residuos en 6 categorías:
- **glass** (Vidrio)
- **metal** (Metal)
- **organic** (Orgánico)
- **others** (Otros)
- **paper** (Papel/Cartón)
- **plastic** (Plástico)

### Instrucciones:
1. Sube tu carpeta `dataset_augmented` a Google Drive
2. Ejecuta todas las celdas en orden
3. Al final descarga `modelo_residuos.h5`

In [ ]:
# Paso 1: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Paso 2: Configuración
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {tf.config.list_physical_devices("GPU")}')

# === CONFIGURACION ===
# Cambia esta ruta si tu dataset está en otra ubicación de Drive
DATASET_PATH = '/content/drive/MyDrive/dataset_augmented'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
CATEGORIAS = ['glass', 'metal', 'organic', 'others', 'paper', 'plastic']

# Verificar que el dataset existe
if os.path.exists(DATASET_PATH):
    for cat in sorted(os.listdir(DATASET_PATH)):
        cat_path = os.path.join(DATASET_PATH, cat)
        if os.path.isdir(cat_path):
            count = len(os.listdir(cat_path))
            print(f'  {cat}: {count} imagenes')
else:
    print(f'ERROR: No se encontro {DATASET_PATH}')
    print('Sube tu carpeta dataset_augmented a Google Drive primero.')

In [ ]:
# Paso 3: Cargar dataset con split 80% train / 20% validation
train_ds = keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    labels='inferred',
    label_mode='categorical',
    class_names=CATEGORIAS,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset='training',
)

val_ds = keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    labels='inferred',
    label_mode='categorical',
    class_names=CATEGORIAS,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset='validation',
)

print(f'\nClases: {train_ds.class_names}')
print(f'Batches de entrenamiento: {len(train_ds)}')
print(f'Batches de validacion: {len(val_ds)}')

# Verificar una muestra
for images, labels in train_ds.take(1):
    print(f'Shape de batch de imagenes: {images.shape}')
    print(f'Shape de batch de labels: {labels.shape}')
    print(f'Rango de pixeles: [{images.numpy().min():.1f}, {images.numpy().max():.1f}]')
    break

In [ ]:
# Paso 4: Optimizar carga de datos
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Paso 5: Construir el modelo con Transfer Learning (MobileNetV2)
#
# IMPORTANTE: El modelo incluye una capa Rescaling que convierte
# pixeles de [0, 255] a [-1, 1] (lo que MobileNetV2 espera).
# Por eso en el backend NO debemos dividir por 255.

# Base model: MobileNetV2 preentrenado en ImageNet (sin la cabeza)
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    pooling=None,
)

# Congelar las capas base inicialmente
base_model.trainable = False

# Construir modelo completo
model = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    # Rescaling: [0, 255] -> [-1, 1] (requerido por MobileNetV2)
    layers.Rescaling(scale=1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CATEGORIAS), activation='softmax'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
# Paso 6: Entrenar Fase 1 — Solo las capas nuevas (base congelada)
callbacks_phase1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

print('=== FASE 1: Entrenando capas nuevas (base congelada) ===')
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_phase1,
)

print(f'\nMejor val_accuracy Fase 1: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
# Paso 7: Fine-tuning — Descongelar las ultimas 30 capas de MobileNetV2
base_model.trainable = True

# Congelar todas excepto las ultimas 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Capas entrenables de MobileNetV2: {trainable_count}/{len(base_model.layers)}')

# Recompilar con learning rate mas bajo para fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint('modelo_residuos_best.h5', monitor='val_accuracy', save_best_only=True),
]

print('\n=== FASE 2: Fine-tuning (ultimas 30 capas de MobileNetV2) ===')
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_phase2,
)

print(f'\nMejor val_accuracy Fase 2: {max(history2.history["val_accuracy"]):.4f}')

In [ ]:
# Paso 8: Evaluar el modelo
import matplotlib.pyplot as plt

# Combinar historiales
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(acc, label='Train Accuracy')
ax1.plot(val_acc, label='Val Accuracy')
ax1.axvline(x=len(history1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-tuning start')
ax1.set_title('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(loss, label='Train Loss')
ax2.plot(val_loss, label='Val Loss')
ax2.axvline(x=len(history1.history['loss'])-1, color='gray', linestyle='--', label='Fine-tuning start')
ax2.set_title('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Evaluacion final
val_loss_final, val_acc_final = model.evaluate(val_ds)
print(f'\nAccuracy final en validacion: {val_acc_final*100:.2f}%')

In [ ]:
# Paso 9: Matriz de confusion
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Obtener predicciones
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Classification report
print(classification_report(y_true, y_pred, target_names=CATEGORIAS))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORIAS, yticklabels=CATEGORIAS)
plt.xlabel('Prediccion')
plt.ylabel('Real')
plt.title('Matriz de Confusion')
plt.tight_layout()
plt.show()

In [ ]:
# Paso 10: Prueba rapida con imagenes del dataset
import random

print('=== Prueba con imagenes aleatorias del dataset ===')
for cat in CATEGORIAS:
    cat_path = os.path.join(DATASET_PATH, cat)
    files = os.listdir(cat_path)
    sample = random.choice(files)
    img_path = os.path.join(cat_path, sample)

    img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = keras.utils.img_to_array(img)  # [0, 255]
    img_array = np.expand_dims(img_array, axis=0)

    pred = model.predict(img_array, verbose=0)
    pred_cat = CATEGORIAS[np.argmax(pred[0])]
    pred_conf = pred[0][np.argmax(pred[0])] * 100

    status = 'OK' if pred_cat == cat else 'FAIL'
    print(f'  [{status}] Real: {cat:10s} -> Prediccion: {pred_cat:10s} ({pred_conf:.1f}%)')

In [ ]:
# Paso 11: Guardar modelo final
model.save('modelo_residuos.h5')
print(f'Modelo guardado: modelo_residuos.h5')
print(f'Tamano: {os.path.getsize("modelo_residuos.h5") / 1048576:.2f} MB')

# Copiar a Google Drive
import shutil
drive_dest = '/content/drive/MyDrive/modelo_residuos.h5'
shutil.copy('modelo_residuos.h5', drive_dest)
print(f'Copiado a Google Drive: {drive_dest}')

In [ ]:
# Paso 12: Descargar modelo (alternativa a Drive)
from google.colab import files
files.download('modelo_residuos.h5')